In [1]:
 !pip install evaluate sacrebleu bitsandbytes datasets transformers accelerate -U -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 79.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 73.1 MB/s eta 0:00:00:00:01


In [2]:
# ── Cell 2: HuggingFace login
from huggingface_hub import notebook_login
notebook_login()

In [3]:
#  Cell 3: Imports 
import os
import torch
import evaluate
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    NllbTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
#  Cell 4: Config
MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"
DATASET_NAME     = "NjeriKahoro/thiomi-multilingual-text"
HUB_REPO_NAME    = "NjeriKahoro/nllb-200-Thiomi-Kik-Eng-2"

LANG_A_COL  = "kikuyu"
LANG_A_CODE = "kik_Latn"
LANG_B_COL  = "english"
LANG_B_CODE = "eng_Latn"

MAX_LENGTH                = 64
BATCH_SIZE                = 1
GRADIENT_ACCUMULATION     = 16

In [5]:
#  Cell 5: Load & clean dataset 
print("Loading dataset...")
raw_datasets = load_dataset(DATASET_NAME)

def filter_empty(example):
    return (
        example[LANG_A_COL] is not None and len(str(example[LANG_A_COL]).strip()) > 0 and
        example[LANG_B_COL] is not None and len(str(example[LANG_B_COL]).strip()) > 0
    )

raw_datasets = raw_datasets.filter(filter_empty)
print(f"Train samples (base pairs): {len(raw_datasets['train'])}")
print(f"Test  samples (base pairs): {len(raw_datasets['test'])}")

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5453 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1365 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5453 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1365 [00:00<?, ? examples/s]

Train samples (base pairs): 5453
Test  samples (base pairs): 1365


In [6]:
# Cell 6: Load model & tokenizer 
print("Loading tokenizer and model...")
tokenizer = NllbTokenizer.from_pretrained(MODEL_CHECKPOINT)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

# FIX 1: Ensure forced_bos is never hardcoded in the saved config
model.config.forced_bos_token_id = None
model.gradient_checkpointing_enable()

# Validate language tokens exist
for code in [LANG_A_CODE, LANG_B_CODE]:
    tid = tokenizer.convert_tokens_to_ids(code)
    assert tid != tokenizer.unk_token_id, f"Token {code} not in vocab!"
    print(f"  {code} → id {tid} ✓")


Loading tokenizer and model...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  kik_Latn → id 256093 ✓
  eng_Latn → id 256047 ✓


In [7]:
#  Cell 7: Preprocessing 
def preprocess_bidirectional(examples):
    inputs, targets, src_langs, tgt_langs = [], [], [], []

    for a, b in zip(examples[LANG_A_COL], examples[LANG_B_COL]):
        # Direction 1: Kikuyu → English
        inputs.append(str(a));   targets.append(str(b))
        src_langs.append(LANG_A_CODE); tgt_langs.append(LANG_B_CODE)
        # Direction 2: English → Kikuyu
        inputs.append(str(b));   targets.append(str(a))
        src_langs.append(LANG_B_CODE); tgt_langs.append(LANG_A_CODE)

    model_inputs = {"input_ids": [], "attention_mask": [], "labels": []}

    for idx in range(len(inputs)):
        tokenizer.src_lang = src_langs[idx]
        tok_in  = tokenizer(inputs[idx],  max_length=MAX_LENGTH, truncation=True)
        tokenizer.src_lang = tgt_langs[idx]
        tok_tgt = tokenizer(targets[idx], max_length=MAX_LENGTH, truncation=True)

        model_inputs["input_ids"].append(tok_in["input_ids"])
        model_inputs["attention_mask"].append(tok_in["attention_mask"])
        model_inputs["labels"].append(tok_tgt["input_ids"])

    return model_inputs

print("Tokenizing dataset (both directions)...")
tokenized_datasets = raw_datasets.map(
    preprocess_bidirectional,
    batched=True,
    batch_size=1000,
    remove_columns=raw_datasets["train"].column_names,
)

# FIX 2: Shuffle so ki→en and en→ki are interleaved, not grouped
tokenized_datasets["train"] = tokenized_datasets["train"].shuffle(seed=42)
tokenized_datasets["test"]  = tokenized_datasets["test"].shuffle(seed=42)

Tokenizing dataset (both directions)...


Map:   0%|          | 0/5453 [00:00<?, ? examples/s]

Map:   0%|          | 0/1365 [00:00<?, ? examples/s]

In [8]:
#  Cell 8: Verify direction balance 
print("\n── Direction balance check ──")
ki_id  = tokenizer.convert_tokens_to_ids(LANG_A_CODE)
en_id  = tokenizer.convert_tokens_to_ids(LANG_B_CODE)

for split in ["train", "test"]:
    first_tokens = [ids[0] for ids in tokenized_datasets[split]["labels"]]
    counts = Counter(first_tokens)
    ki_count = counts.get(ki_id, 0)
    en_count = counts.get(en_id, 0)
    total    = len(first_tokens)
    print(f"  {split}: ki→en={en_count} ({en_count/total*100:.1f}%)  "
          f"en→ki={ki_count} ({ki_count/total*100:.1f}%)")


── Direction balance check ──
  train: ki→en=5453 (50.0%)  en→ki=5453 (50.0%)
  test: ki→en=1365 (50.0%)  en→ki=1365 (50.0%)


In [9]:
#  Cell 9: Metrics 
chrf_metric = evaluate.load("chrf")
bleu_metric = evaluate.load("bleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    preds  = np.where(preds  != -100, preds,  pad_id)
    labels = np.where(labels != -100, labels, pad_id)

    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [[l.strip()] for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]

    return {
        "chrf": chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"],
        "bleu": bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)["bleu"] * 100,
    }

In [10]:
#  Cell 10: Training arguments 
training_args = Seq2SeqTrainingArguments(
    output_dir=HUB_REPO_NAME,

    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",          # FIX 3: save every epoch
    load_best_model_at_end=True,    # FIX 4: keep best checkpoint
    metric_for_best_model="chrf",
    greater_is_better=True,

    # Learning schedule
    learning_rate=2e-5,
    warmup_ratio=0.1,               # FIX 5: warmup over 10% of steps
    weight_decay=0.01,
    num_train_epochs=5,             # FIX 6: more epochs (was 3)

    # VRAM
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    fp16=True,

    # Generation & logging
    predict_with_generate=True,
    logging_steps=50,
    report_to="none",

    # Hub
    push_to_hub=True,
    hub_model_id=HUB_REPO_NAME,
    hub_strategy="end",
    hub_private_repo=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
#  Cell 11: Train 
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # FIX 7
)

print("Starting bidirectional training...")
trainer.train()

Starting bidirectional training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Chrf,Bleu
1,54.562979,1.583848,45.578381,19.586288
2,50.630908,1.536096,46.638028,20.288722


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [ ]:
#  Cell 12: Save & push to Hub 
print("Pushing best model to Hub...")

# FIX 8: Explicitly clear forced_bos before pushing so Hub config is clean
trainer.model.config.forced_bos_token_id = None

trainer.push_to_hub(
    commit_message="Retrain v2: balanced bidirectional, warmup, best-model saving, forced_bos=null",
    finetuned_from=MODEL_CHECKPOINT,
    tasks="translation",
    dataset=DATASET_NAME,
    tags=["translation", "nllb", "kikuyu", "bidirectional"],
    language=["ki", "en"],
)
tokenizer.push_to_hub(HUB_REPO_NAME)
print("Upload complete ✓")

In [ ]:
#  Cell 13: Sanity-check inference 
print("\n── Inference sanity check ──")

def translate(text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=MAX_LENGTH).to(trainer.model.device)
    # FIX 9: use convert_tokens_to_ids — NOT lang_code_to_id (doesn't exist on NllbTokenizer)
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    out = trainer.model.generate(
        **inputs,
        forced_bos_token_id=forced_bos,
        max_length=128,
        num_beams=4,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

tests = [
    ("Mbura nene na riũwa inini?",  LANG_A_CODE, LANG_B_CODE, "ki→en"),
    ("Rũciũ rũrĩa nĩ rũega.",       LANG_A_CODE, LANG_B_CODE, "ki→en"),
    ("It's a good day today.",       LANG_B_CODE, LANG_A_CODE, "en→ki"),
    ("The child is playing outside.", LANG_B_CODE, LANG_A_CODE, "en→ki"),
]

for text, src, tgt, label in tests:
    result = translate(text, src, tgt)
    print(f"  [{label}] {text}")
    print(f"         → {result}\n")

In [ ]:
#  Cell 14: Training curves 
history        = trainer.state.log_history
train_log_data = [l for l in history if "loss"      in l and "eval_loss" not in l]
eval_log_data  = [l for l in history if "eval_loss" in l]
df_train = pd.DataFrame(train_log_data)
df_eval  = pd.DataFrame(eval_log_data)

os.makedirs(HUB_REPO_NAME, exist_ok=True)
df_train.to_csv(f"{HUB_REPO_NAME}/training_logs.csv",   index=False)
df_eval.to_csv( f"{HUB_REPO_NAME}/evaluation_logs.csv", index=False)

sns.set_theme(style="whitegrid")

# Loss curve
plt.figure(figsize=(10, 5))
if not df_train.empty:
    plt.plot(df_train["epoch"], df_train["loss"],      label="Train Loss",  color="royalblue", alpha=0.7, lw=2)
if not df_eval.empty:
    plt.plot(df_eval["epoch"],  df_eval["eval_loss"],  label="Val Loss",    color="crimson",   marker="o", lw=2)
plt.title("NLLB Fine-Tuning: Loss Progression", fontsize=14, fontweight="bold")
plt.xlabel("Epoch"); plt.ylabel("Cross-Entropy Loss")
plt.legend(); plt.tight_layout()
plt.savefig(f"{HUB_REPO_NAME}/loss_curves.png", dpi=300); plt.close()

# BLEU / ChrF curve
if not df_eval.empty and "eval_bleu" in df_eval.columns:
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("BLEU", color="darkorange")
    ax1.plot(df_eval["epoch"], df_eval["eval_bleu"], color="darkorange", marker="s", lw=2, label="BLEU")
    ax1.tick_params(axis="y", labelcolor="darkorange")
    ax2 = ax1.twinx()
    ax2.set_ylabel("ChrF", color="teal")
    ax2.plot(df_eval["epoch"], df_eval["eval_chrf"], color="teal", marker="^", lw=2, ls="--", label="ChrF")
    ax2.tick_params(axis="y", labelcolor="teal")
    lines  = ax1.get_lines() + ax2.get_lines()
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc="upper left")
    plt.title("Evaluation Metrics: BLEU & ChrF", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{HUB_REPO_NAME}/generation_metrics.png", dpi=300); plt.close()

print("All plots saved ✓")